# Lazypredict

In [1]:
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

In [2]:
# Data Preparation
df = pd.read_excel("Final_PM15-1.xlsx")

# Cleaning
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[df['GSM'].str.len() <= 4].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)

# Create Pseudo_Mass Feature
df['pseudo_mass'] = (df['Mean_Stock Flow'] * df['Mean_Stock Consistency'])/ df['Mean_Yankee Speed']

# Create Coating-Release Ratio Feature
df['coating_release_ratio'] = df['Mean_Flow Coating'] / df['Mean_Flow Release']

# X Variables
features = [
    '% NBKP',
    'Mean_Load KWH Tickling Refiner',
    'Mean_Creping',
    'Mean_Jet Wire Ratio',
    'GSM',
    'coating_release_ratio'
]
X = df[features]

# Y Variables

y = df['MDT']

In [3]:
X.tail()

,% NBKP,Mean_Load KWH Tickling Refiner,Mean_Creping,Mean_Jet Wire Ratio,GSM,coating_release_ratio
1068,0.0,320.397500,23.006510,0.94,16.0,0.632760
1069,0.0,314.134920,23.489669,0.94,16.0,0.652042
1070,0.0,311.355421,23.135163,0.94,16.0,0.632765
1071,0.0,313.115333,22.726395,0.94,16.0,0.632764
1072,0.0,314.591462,21.770716,0.94,16.0,0.632761


In [4]:
print(X['coating_release_ratio'].max())
print(X['coating_release_ratio'].min())

1.0546138450386158
0.5316963625663452


In [5]:
y.head()

21    1794
22    1977
23    1739
24    1693
25    1686
Name: MDT, dtype: int64

In [6]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

In [8]:
# Hitung MAPE untuk tiap model
mape_scores = {}

for model_name, y_pred in predictions.items():
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mape_scores[model_name] = mape

# Tambahkan ke tabel hasil
models["MAPE"] = pd.Series(mape_scores)

# Urutkan (semakin kecil semakin baik)
models = models.sort_values(by="MAPE")

In [9]:
print(models)

                               Adjusted R-Squared   R-Squared         RMSE  \
Model                                                                        
ExtraTreesRegressor                      0.976681    0.978289    56.389488   
GradientBoostingRegressor                0.972641    0.974528    61.078271   
RandomForestRegressor                    0.971808    0.973752    62.001722   
HistGradientBoostingRegressor            0.967527    0.969766    66.543072   
KNeighborsRegressor                      0.966872    0.969156    67.210899   
ExtraTreeRegressor                       0.955671    0.958728    77.746912   
BaggingRegressor                         0.953816    0.957002    79.356626   
AdaBoostRegressor                        0.950369    0.953791    82.265546   
PoissonRegressor                         0.948230    0.951800    84.019375   
DecisionTreeRegressor                    0.944709    0.948522    86.829525   
SGDRegressor                             0.934304    0.938835   

# Regressor

In [10]:
# Import Libraries
import numpy as np

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error)

# Model Dasar
model = ExtraTreesRegressor(random_state=42,n_jobs=-1)

# Grid Search Hyper Parameter
param_grid = {'n_estimators': [100, 200, 300],
              'max_depth': [5, 10, 15],
              'min_samples_split': [2, 5, 10],
              'min_samples_leaf': [1, 2, 4],
              'max_features': ['sqrt']
            }

# K-Fold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# GGrid Search CV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

# Training + Tuning
grid_search.fit(X, y)

# Model Terbaik
best_model = grid_search.best_estimator_
print("Best Parameters:")
print(grid_search.best_params_)

# Prrediksi
y_pred = best_model.predict(X)

# Matrik Evaluasi
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
epsilon = 1e-8
mape = np.mean(np.abs((y - y_pred) / (y + epsilon))) * 100

# Hasil
print("\n===== HASIL MODEL TERBAIK =====")
print(f"R2    : {r2:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"MAE   : {mae:.4f}")
print(f"MAPE  : {mape:.2f}%")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best Parameters:
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}

===== HASIL MODEL TERBAIK =====
R2    : 0.9908
RMSE  : 46.8013
MAE   : 34.8180
MAPE  : 4.29%


### Pickle Files

In [11]:
import joblib

In [12]:
#joblib.dump(model, 'model.pkl')

In [13]:
features_pkl = X.columns.tolist()
#joblib.dump(features_pkl, 'features.pkl')

### Features Importance

In [14]:
# Feature Importance
import pandas as pd

feat_imp = pd.DataFrame({
    "Feature": features, 
    "Importance": model.feature_importances_
})

# Urutkan dari terbesar
feat_imp = feat_imp.sort_values(by="Importance", ascending=False)


# Plot
import matplotlib.pyplot as plt
plt.figure()
plt.barh(feat_imp["Feature"], feat_imp["Importance"])
plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Random Forest")

plt.tight_layout()
plt.show()

NotFittedError: This ExtraTreesRegressor instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

### SHAP Analysis

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

In [ ]:
# Scatter plot dasar - melihat tren arah secara mendetail
shap.plots.scatter(shap_values[:, "Mean_Load KWH Tickling Refiner"])

In [ ]:
shap.plots.scatter(shap_values[:, "pseudo_mass"])

In [ ]:
shap.plots.scatter(shap_values[:, "% NBKP"])

In [ ]:
shap_values = explainer.shap_values(np.array(X_test))
shap.initjs()
i = 0  # index data

shap.force_plot(
    explainer.expected_value,
    shap_values[i],
    X_test.iloc[i]
)